# Figure 3D: implicit vs explicit solvent corrections, one box per solvent

All 12 solvents individually (polar aprotic -> polar protic -> aromatic), four schemes each: implicit (PCM), implicit + vibrations, explicit, explicit + vibrations.

In [ ]:
import os, sys

# make the in-repo modules importable (not pip-installed)
REPO = os.path.abspath("../..")
for _p in ("data/delta22", "analysis/code", "analysis/code/shared"):
    sys.path.insert(0, os.path.join(REPO, _p))

In [ ]:
import delta22
import paths
import fig3_plots

In [ ]:
DELTA22_HDF5 = paths.dataset_file("delta22", root=REPO)
XLSX = os.path.join(REPO, "data", "delta22", "delta22_experimental.xlsx")

def figure_path(name):
    os.makedirs("figures", exist_ok=True)
    return os.path.join("figures", name)

In [ ]:
# the published panels use 250 seeded train/test splits
N_SPLITS = 250

In [ ]:
query = delta22.add_composite_columns(delta22.load_query_df_dft(DELTA22_HDF5, XLSX, verbose=False))
solutes = sorted(query["solute"].unique())
print(len(query), "rows;", len(solutes), "solutes;", query["sap_nmr_method"].nunique(), "methods")

In [ ]:
FIG3D_LABELS = {
    "stationary + pcm": "Implicit Solvent (PCM)",
    "stationary_plus_qcd + pcm": "Implicit Solvent (PCM) + Vibrations",
    "stationary + desmond": "Explicit Solvent",
    "stationary_plus_qcd + desmond": "Explicit Solvent + Vibrations",
}
fig3d_results = delta22.fig3d_formula_regressions(
    query, "dsd_pbep86", "pcSseg3", "pbe0_tz", list(FIG3D_LABELS),
    delta22.DESMOND_SOLVENTS, n_splits=N_SPLITS, solutes=solutes)

# report implicit vs explicit median test RMSE per solvent (not pooled by class)
for solvent in delta22.DESMOND_SOLVENTS:
    sub = fig3d_results[fig3d_results["solvent"] == solvent]
    mi = sub[sub["formula"] == "stationary + pcm"]["test_RMSE"].median()
    me = sub[sub["formula"] == "stationary + desmond"]["test_RMSE"].median()
    print(f"{solvent:20s} implicit={mi:.4f}  explicit={me:.4f}")

In [ ]:
FIG3D_COLORS = ["#A72608", "#F4BAAD", "#5D737E", "#D9FFF5"]   # dark red, pink, gray, mint (as published)
FIG3D_SOLVENT_LABELS = {
    "chloroform": r"CDCl$_3$", "dichloromethane": "DCM", "tetrahydrofuran": "THF",
    "acetonitrile": "MeCN", "dimethylsulfoxide": "DMSO", "methanol": "MeOD",
    "trifluoroethanol": "TFE", "chlorobenzene": "PhCl",
}
# solvent order: polar aprotic -> polar protic -> aromatic (group ordering from delta22.SOLVENT_GROUPS)
FIG3D_SOLVENT_ORDER = [s for group in delta22.SOLVENT_GROUPS.values() for s in group]

In [ ]:
fig3_plots.plot_solvent_correction_boxplot(fig3d_results, FIG3D_LABELS, FIG3D_SOLVENT_ORDER,
                                           FIG3D_COLORS, FIG3D_SOLVENT_LABELS,
                                           save_path=figure_path("fig3d_explicit_solvation_1H.png"))